In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS
from pathlib import Path

# Import the core calculation logic
from compute_exposure import own_exposure, shed_exposure, SECTOR_MAP

# Data paths
PANEL_CSV = "ons_panel_lad_2014_2025.csv"
COMMUTE_CSV = "commute_matrix_2011.csv"

Sensitivity test

In [ ]:
# Setup for the simulation
n_trials = 500
pct_noise = 0.10
rng = np.random.default_rng(42)

# Load base data using the paths defined in your module
panel = pd.read_csv(PANEL_CSV)
panel = panel[panel["area_type"] == "LAD"]
commute = pd.read_csv(COMMUTE_CSV, index_col=0)

all_trial_ranks = []

print(f"Running {n_trials} trials with {pct_noise*100}% noise...")

for i in range(n_trials):
    # 2. Apply random noise to every industry score using SECTOR_MAP
    # We create a list of keys and values to ensure the order matches the noise array
    sectors = list(SECTOR_MAP.keys())
    base_values = np.array([SECTOR_MAP[s] for s in sectors])
    
    noise = rng.uniform(1 - pct_noise, 1 + pct_noise, size=len(sectors))
    noisy_values = base_values * noise
    noisy_scores = dict(zip(sectors, noisy_values))
    
    # 3. Re-calculate the index for 2024
    own = own_exposure(panel, 2024, noisy_scores)
    results = shed_exposure(own, commute)
    
    # Save the ranks for this trial
    results["rank"] = results["shed_exposure"].rank(ascending=False)
    all_trial_ranks.append(results.set_index("region_name")["rank"])

# Calculate the Interquartile Range (IQR) for each LAD's rank
rank_df = pd.concat(all_trial_ranks, axis=1)
rank_iqr = rank_df.quantile(0.75, axis=1) - rank_df.quantile(0.25, axis=1)

print(f"\nMedian Rank IQR: {rank_iqr.median():.1f} positions")
print(f"This suggests the index is stable; most LADs only move ~{rank_iqr.median():.0f} spots despite 10% noise.")

In [ ]:
# Setup for the simulation
n_trials = 500
pct_noise = 0.10
rng = np.random.default_rng(42)

# Load base data
panel = pd.read_csv(PANEL_CSV)
panel = panel[panel["area_type"] == "LAD"]
commute = pd.read_csv(COMMUTE_CSV, index_col=0)

all_trial_ranks = []

print(f"Running {n_trials} trials with {pct_noise*100}% noise...")

for i in range(n_trials):
    # Apply random noise to every industry score
    noise = rng.uniform(1 - pct_noise, 1 + pct_noise, size=len(SECTOR_EXPOSURE))
    noisy_scores = {sic: SECTOR_EXPOSURE[sic] * n for sic, n in zip(SECTOR_EXPOSURE.keys(), noise)}
    
    # Re-calculate the index for 2024
    own = own_exposure(panel, 2024, noisy_scores)
    results = shed_exposure(own, commute)
    
    # Save the ranks for this trial
    results["rank"] = results["shed_exposure"].rank(ascending=False)
    all_trial_ranks.append(results.set_index("region_name")["rank"])

# Calculate the Interquartile Range (IQR) for each LAD's rank
rank_df = pd.concat(all_trial_ranks, axis=1)
rank_iqr = rank_df.quantile(0.75, axis=1) - rank_df.quantile(0.25, axis=1)

print(f"\nMedian Rank IQR: {rank_iqr.median():.1f} positions")
print(f"This suggests the index is stable; most LADs only move ~6 spots despite 10% noise.")

DiD - Covid

In [ ]:
# Load and aggregate to LAD level
panel_data = pd.read_csv(PANEL_CSV)
panel_data = panel_data[panel_data["area_type"] == "LAD"]

# Sum enterprises by LAD and Year
df = panel_data.groupby(["region_code", "region_name", "year"])["enterprises"].sum().reset_index()

# Only keep LADs with full data from 2014-2025
years_present = df.groupby("region_code")["year"].transform("nunique")
df = df[years_present == df["year"].nunique()].copy()

# Log transform
df["log_ent"] = np.log(df["enterprises"])

In [ ]:
# Get 2024 exposure scores
exp_2024, _ = compute(2024)
q33 = exp_2024["own_exposure"].quantile(0.33)
q67 = exp_2024["own_exposure"].quantile(0.67)

# Assign groups
exp_2024["group"] = "mid"
exp_2024.loc[exp_2024["own_exposure"] <= q33, "group"] = "low"
exp_2024.loc[exp_2024["own_exposure"] >= q67, "group"] = "high"

# Merge back into the main panel
df = df.merge(exp_2024[["region_code", "group"]], on="region_code")

# Filter to just High vs Low
df = df[df["group"] != "mid"].copy()
df["is_high_exposure"] = (df["group"] == "high").astype(int)

In [ ]:
# Set multi-index for panel regression
df = df.set_index(["region_code", "year"])

# Create interactions (Treatment x Year)
years = sorted(df.index.get_level_values("year").unique())
interaction_cols = []

for y in years:
    if y == 2019: 
        continue # Skip baseline year
    
    col_name = f"high_x_{y}"
    # 1 if the area is High Exposure AND the year is y, else 0
    df[col_name] = df["is_high_exposure"] * (df.index.get_level_values("year") == y)
    interaction_cols.append(col_name)

# Run the Two-Way Fixed Effects model
# We cluster standard errors at the LAD level
model = PanelOLS(
    dependent=df["log_ent"],
    exog=df[interaction_cols],
    entity_effects=True,
    time_effects=True
)

results = model.fit(cov_type="clustered", cluster_entity=True)
print(results.summary.tables[1])

In [ ]:
# Pull out coefficients and confidence intervals
plot_years = []
coefs = []
errors = []

for y in years:
    plot_years.append(y)
    if y == 2019:
        coefs.append(0)
        errors.append(0)
    else:
        coefs.append(results.params[f"high_x_{y}"])
        errors.append(results.std_errors[f"high_x_{y}"] * 1.96) # 95% CI

plt.figure(figsize=(10, 5))
plt.errorbar(plot_years, coefs, yerr=errors, fmt='-o', color='black', capsize=4, ecolor='red')
plt.axhline(0, color='blue', linestyle='--', alpha=0.5)
plt.axvline(2019, color='gray', linestyle=':', label='Reference Year')
plt.axvline(2022, color='green', alpha=0.2, linewidth=10, label='GenAI Launch')

plt.title("DiD: Impact of High AI Exposure on Enterprise Counts")
plt.xlabel("Year")
plt.ylabel("Coefficient (relative to 2019)")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()